In [ ]:
import random 
import string
import requests
import csv
import numpy as np
from datetime import datetime, UTC
import time
import langid
import pandas as pd

### Choisir un fichier `covid_user_X` à scraper
Ce $X$ détermine également dans quel fichier `data_X.csv` les données seront sauvegardées.

In [ ]:
X = 1

Charger les noms d'utilisateurs dont récupérer les messages

In [ ]:
with open(f'covid_users_{X}.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

df = pd.read_csv(f'data_{X}.csv', sep=',')
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(by=['user', "date"], ascending=True)

### Récupération des messages
Avec ce premier script, nous essayons de réduire le nombre de contributions nécessaires pour remplir le jeu de données. Nous récupérons les contributions par nom d’utilisateur par lots de 50 requêtes. Ensuite, pour chaque contribution, nous effectuons 1 requête supplémentaire (le post sous lequel l'utilisateur commente) ou 2 requêtes supplémentaires (post + contribution de l’interlocuteur, dans le cas d’une réponse à un commentaire).

Après chaque requête, les résultats sont sauvegardés dans `data.csv` avec un le paramètre url de la page suivante à requêter pour le même utilisateur, ou la mention "End" si toutes les interactions de l'utilisateur ont été récupérées. 

Nous implémentons également un test pour filtrer les utilisateurs non-francophone (qui sont par exemple passé sur r/france pour poser une question touristique). Après la première requête (50 contributions), nous écartons les utilisateurs qui ont interagi 5 fois ou moins en français. La langue est estimée avec le package python langid.

Nous avons initialement laissé des vides pour la date de création du compte et la photo de profil de l'utilisateur, mais nous ne les avons finalement pas remplies (la première est moins informative avec la date de la première publication, et la deuxième trop peu informative $-$ beaucoup de photos de profil génériques). 

In [ ]:
random_sleeps = np.random.uniform(low=0.35, high=0.65, size=100_000_000)

for i, username in enumerate(lines):
    username = username.strip('\n')
    
    after, to_cont = None, True
    j = (df['user']==username).sum() + 1

    # If username already in dataframe, load last 'after' property
    if j > 1:
        subset = df[df['user']==username]
        if 'End' in subset['after'].unique():
            to_cont = False
        after = subset.iloc[0]['after']
    else : 
        time.sleep(15+random_sleeps[i]) # sleep for new usernames

    print(username)
    
    headuser = "Mozilla/5.0 (compatible; scraper/1.0)"
    

    # Loop until all contributions have been scraped
    while to_cont:
        headuser += random.choice(string.ascii_uppercase + string.digits)
        headers = {
            "User-Agent": headuser
        }
        session = requests.Session()
        session.headers.update(headers)
        params = params = {
                "limit": 50
            }
        
        if after:
            params["after"] = after

        try :
            r = session.get(
                    f"https://www.reddit.com/user/{username}/.json",
                    params=params,
                    timeout=15)
            data = r.json()
        except Exception:
            to_cont = False
            continue

        if 'data' in data.keys() and data["data"]["after"]:
            after = data["data"]["after"]
        else : 
            after = 'End'
            to_cont = False

        # For every contribution in the request, gather relevant informations
        if 'data' in data.keys() and data['data']['children']:
            donnees_user = []
            batch_language = []
            for child in data['data']['children']:
                id = str(i+1).zfill(4) + str(j)
                prod = child['data']
                prod_created = datetime.fromtimestamp(prod['created_utc'], UTC)

                if child['kind']=='t1': # contribution = comment 

                    # gather information on the post
                    post_title = prod['link_title']
                    while True : 
                        try :
                            r = session.get(
                                    f"https://www.reddit.com/api/info.json?id={prod['link_id']}",
                                    timeout=3)
                            post = r.json()['data']['children'][0]['data']
                            post_body = "".join(c for c in post['selftext'] if c.isprintable())
                            parent_user = post['author']
                            break
                        except : 
                            print("Error. Retrying later.")
                            time.sleep(3+random_sleeps[i+1000+j])

                    body = "".join(c for c in prod['body'] if c.isprintable())

                    if prod['parent_id'].startswith('t1'): # comment to comment (= response)
                        type_prod = 'response'
                        while True:
                            try:
                                r = session.get(
                                f"https://www.reddit.com/api/info.json?id={prod['parent_id']}",
                                timeout=3)
                                parent = r.json()['data']['children'][0]['data']
                                parent_user = parent['author']
                                parent_body = "".join(c for c in parent['body'] if c.isprintable())
                                break  # sort de la boucle si succès
                            except Exception:
                                print("Error. Retrying later.")
                                time.sleep(3+random_sleeps[i+1000-j])

                    elif prod['parent_id'].startswith('t3') : # Comment to post (=comment)
                        type_prod = 'comment'
                        parent_body = 'post body'
                    

                elif child['kind']=='t3': # contribution = post
                    type_prod = 'post'
                    body = "".join(c for c in prod['selftext'] if c.isprintable())
                    parent_user = np.nan
                    parent_body = np.nan
                    post_title = prod['title']
                    post_body = 'self body'
                
                # Estimate if the contribution is in french
                language, score = langid.classify(body)
                batch_language.append(language)

            
                interaction = [f'{X}X{id}', username, np.nan, type_prod, prod_created,
                    body, parent_user,  np.nan, parent_body, 
                    post_title, post_body, after, language] 
                # np.nan correspondent respectivement à la date de création du compte de l'utilisateur 
                # ainsi qu'à la photo de profil de l'interlocuteur.
                donnees_user.append(interaction)

                j += 1
                if j%100 == 0:
                    time.sleep(random_sleeps[j])
                    print(j, body)

            # If few contributions are in english in first batch, then discard user
            if j<=51:
                if batch_language.count('fr') <=5:
                    with open(f"data_{X}.csv", "a", newline="", encoding="utf-8") as f:
                        writer = csv.writer(f)
                        writer.writerow([f'{X}X{id}', username] + [np.nan for k in range(9)] + ['End', 'en'])  
                    to_cont = False
                    print('Non French found')
                    continue

            # Save on data.csv
            with open(f"data_{X}.csv", "a", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerows(donnees_user)

    print(f'{i+1} user(s) done.')